In [ ]:
# === Setup ===
# Runtime: <1m with OAI_FAST_MODE=1
# Hardware: CPU smoke
# Network: none
# Competition-safe: Yes for the declared profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
import torch
torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
DEVICE = torch.device("cuda" if RUNTIME_PROFILE == "gpu" else "cpu")
if RUNTIME_PROFILE == "gpu" and not torch.cuda.is_available(): raise RuntimeError("GPU profile requested but CUDA is unavailable")
torch.set_default_device(DEVICE)
_device_probe = (torch.ones(8, device=DEVICE) @ torch.ones(8, device=DEVICE)).item()
print(f"Compute device: {DEVICE}; probe={_device_probe:.1f}")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# Experiment — stride và padding

**Hypothesis:** stride giảm spatial resolution; padding giữ biên.

In [ ]:
import torch.nn.functional as F
x=torch.ones(1,1,8,8); k=torch.ones(1,1,3,3)
for stride,pad in [(1,0),(1,1),(2,1)]: print(stride,pad,tuple(F.conv2d(x,k,stride=stride,padding=pad).shape))

**Observation:** kiểm tra công thức `floor((H+2P-K)/S)+1` cho từng cấu hình.